In [1]:
# -----------------------------
# Step 0: Imports
# -----------------------------
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import numpy as np
import os

# -----------------------------
# Step 1: Encode labels
# -----------------------------
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])
print("✅ Labels encoded:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

# -----------------------------
# Step 2: Subset dataset (20%)
# -----------------------------
subset_frac = 0.2
df_subset = df.sample(frac=subset_frac, random_state=42)
print(f"✅ Using {len(df_subset)} samples ({subset_frac*100}% of original dataset)")

# -----------------------------
# Step 3: Train/Val/Test Split
# -----------------------------
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df_subset["text"].tolist(),
    df_subset["label_id"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_subset["label_id"]
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

print(f"✅ Split: {len(train_texts)} train | {len(val_texts)} val | {len(test_texts)} test")

# -----------------------------
# Step 4: Tokenization with progress
# -----------------------------
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def batch_tokenize(texts, batch_size=1000, max_length=128):
    input_ids, attention_masks = [], []
    for i in tqdm(range(0, len(texts), batch_size), desc="Tokenizing"):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding="max_length", truncation=True, max_length=max_length, return_tensors="pt")
        input_ids.append(enc["input_ids"])
        attention_masks.append(enc["attention_mask"])
    return {
        "input_ids": torch.cat(input_ids, dim=0),
        "attention_mask": torch.cat(attention_masks, dim=0)
    }

# Check if tokenized dataset exists
if not os.path.exists("bert_tokenized"):
    os.makedirs("bert_tokenized")
    
train_encodings = batch_tokenize(train_texts)
val_encodings   = batch_tokenize(val_texts)
test_encodings  = batch_tokenize(test_texts)

torch.save(train_encodings, "bert_tokenized/train_encodings.pt")
torch.save(val_encodings, "bert_tokenized/val_encodings.pt")
torch.save(test_encodings, "bert_tokenized/test_encodings.pt")
torch.save(torch.tensor(train_labels), "bert_tokenized/train_labels.pt")
torch.save(torch.tensor(val_labels), "bert_tokenized/val_labels.pt")
torch.save(torch.tensor(test_labels), "bert_tokenized/test_labels.pt")

print("✅ Tokenization done and saved to disk!")

# -----------------------------
# Step 5: Create PyTorch datasets
# -----------------------------
class RedditDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = RedditDataset(train_encodings, torch.tensor(train_labels))
val_dataset   = RedditDataset(val_encodings, torch.tensor(val_labels))
test_dataset  = RedditDataset(test_encodings, torch.tensor(test_labels))

print("✅ PyTorch datasets ready!")

# -----------------------------
# Step 6: Load DistilBERT model
# -----------------------------
num_labels = len(label_encoder.classes_)
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)
print(f"✅ DistilBERT model loaded for {num_labels} labels!")

# -----------------------------
# Step 7: Define metrics
# -----------------------------
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# -----------------------------
# Step 8: Training arguments
# -----------------------------
training_args = TrainingArguments(
    output_dir="./results_distilbert",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir="./logs_distilbert",
    logging_steps=200,
    save_steps=500,
    save_total_limit=2,
    evaluation_strategy="steps",
    eval_steps=500,
    load_best_model_at_end=True,
    fp16=True if torch.cuda.is_available() else False
)
print("✅ Training arguments set!")

# -----------------------------
# Step 9: Trainer
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)
print("✅ Trainer ready!")

# -----------------------------
# Step 10: Train
# -----------------------------
trainer.train()
print("✅ Training complete!")

# -----------------------------
# Step 11: Evaluate on test set
# -----------------------------
results = trainer.evaluate(test_dataset)
print("✅ Test Results:", results)


NameError: name 'df' is not defined